<img src="images/m-rainbow.svg" width="5%" height="5%">

<h1 style="font-size: 30px; font-weight: bold; color: #ff2f05;">
  The Mistral AI Python SDK
</h1>

The Mistral AI Python SDK (**S**oftware **D**evelopment **K**it) is a wrapper for the **Mistral AI API**.

You can find the official documentation and some examples in:
- The [Vibe Studio Product Section](https://docs.mistral.ai/studio-api/overview) 
- The [API reference](https://docs.mistral.ai/api)
- The [Developers Section](https://docs.mistral.ai/developers)
- Their Github [Python SDK](https://github.com/mistralai/client-python) and [Cookbook](https://github.com/mistralai/cookbook) repositories
- Their [YouTube Streams](https://www.youtube.com/@MistralAIOfficial/streams)

<h2 style="font-size: 25px; font-weight: bold; color: #fb6227;">
  3. Working with a Local Model
</h2>

Mistral AI is well-known for their contribution to open-source models: you can download, fine-tune and run some of their models on your own hardware.

You can keep your data private and save on token costs.

**Objectives**: Use a locally-running model with the Mistral Python SDK instead of the regular API. Depending on the model, it may require a powerful computer.

**Difficulty**: 🟢 ⚪️ ⚪️ ⚪️ - Only some basic Python knowledge is required (imports, lists, dict, loops, etc). 

---
<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  3.1 Install LM Studio and Download Model
</h3>

I used the following [video from WEBdoze](https://www.youtube.com/watch?v=3ESDoDlzb80) to get started with LM Studio. Check it out for additional details.

1. Go to the [LM Studio Website](https://lmstudio.ai/) to download and install their desktop application.
2. Download the smallest ministral model (3B parameter).

<img src="images/lm_studio_model.png" width="50%" height="50%">

3. In settings, increase the model context window to be able to perform meaningful tasks

<img src="images/context_window.png" width="40%" height="40%">

4. Load the model and start the server on port 1234

<img src="images/lm_studio_server.png" width="30%" height="30%">

5. Copy the server URL

<img src="images/server_url.png" width="30%" height="30%">

6. Your Mistral client requires the extra `server_url` parameter and a dummy api_key to function locally:



In [9]:
from mistralai.client import Mistral
from mistralai.client.models import UserMessage

client = Mistral(
    api_key="lm-studio",                 # A dummy API key is required by the SDK
    server_url="http://localhost:1234"   # Paste the server URL here (remove the '/v1' as the Mistral client automatically appends '/v1/chat/completions'; a major difference compared to the OpenAI client)
)

chat_response = client.chat.complete(
    model="mistralai/ministral-3-3b" ,   # Paste the full model name here as shown in LM Studio
    messages=[UserMessage(content="Explain the difference between a local LLM and a cloud LLM in one sentence.")]
)

print(chat_response.choices[0].message.content)

A **local LLM** runs on your own device (like a smartphone or PC) using offline data, while a **cloud LLM** processes requests through remote servers connected to the internet for faster responses but requires an active connection.


<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  3.2 Use Local Model on a Classification Task
</h3>

In the example below, we use a [Kaggle dataset](https://www.kaggle.com/datasets/andrewmvd/trip-advisor-hotel-reviews) about hotel reviews. <br>
*Alam, M. H., Ryu, W.-J., Lee, S., 2016. Joint multi-grain topic sentiment: modeling semantic aspects for online reviews. Information Sciences 339, 206–223*

This dataset has been preprocessed for better accuracy (e.g. removed stop words, everything in lower case, etc).

We will supply a sample of the reviews to the local model and ask it to use the most relevant class: Excellent, Good, Neutral, Frustrated or Terrible

In [10]:
import pandas as pd
df = pd.read_csv(r"datasets/tripadvisor_hotel_reviews.csv")
sample = df.sample(10)
sample.head()


,Review,Rating
10634,"lovely hotel stayed 3 nights aprile loved, roo...",5
2746,"great place, just got dr stay bavaro beach res...",5
12952,"ola, say, visit resort maybe expectations diff...",4
17002,home away home just returned day visit new orl...,5
10810,not better apprehensive photos room old fashio...,5


In [11]:
from mistralai.client.models import SystemMessage
import time

outcome = []
start_time = time.time() 

for index, row in sample.iterrows():

    response = client.chat.complete(
        model="mistralai/ministral-3-3b" ,
        messages=[
            SystemMessage(content="You are in charge of sentiment analysis for hotel reviews. Describe the following review in ONE WORD ONLY: Excellent, Good, Neutral, Frustrated or Terrible"),
            UserMessage(content=row["Review"])
        ]
    )

    outcome.append(
        {
            "Review": row["Review"], 
            "Sentiment": response.choices[0].message.content
        }
    )

# For context, this ran on a Macbook Pro M4 with 24GB of memory
elapsed = time.time() - start_time
print(f"Total time: {elapsed:.2f} seconds for {len(sample)} reviews")

Total time: 9.59 seconds for 10 reviews


In [12]:
outcome

[{'Review': 'lovely hotel stayed 3 nights aprile loved, room fairly small clean comfortable, bathroom looked newly fitted.breakfast good nice choice excellent service, square hotel not nicest places felt uncomfortable walking night,  ',
  'Sentiment': '**Neutral**'},
 {'Review': 'great place, just got dr stay bavaro beach resort great, pretty picky not like place, stay paid drinks food, little concerned reading comments site food fine no group got sick, people spoke english way problems traveled non-english speaking countries people really need communicate excursion planners desk restaurant staff spoke english, lot guests resort european met quite people u.s. couple tips fly punta cana airport easier drinks weak simply ask alcohol staff willing help shows night massage beach tip staff couple dollars appreciated,  ',
  'Sentiment': '**Neutral**'},
 {'Review': 'ola, say, visit resort maybe expectations different like people went, got airport little straw hut basically, bus ride took fore

<h3 style="font-size: 20px; font-weight: bold; color: #ff8f1e;">
  3.3 Connect the Local Model to your Vibe VSCode Extension
</h3>

You can also connect your Vibe coding assistant in VS Code to a local model.
The example below is still based on the 3B model but it is recommended to use a larger model if possible.


1. Open the config file located at `~/.vibe/config.toml` on MacOS (use command+Shift+. to display hidden folders)
2. Add a provider and model block like below (with the right URL, including the /v1, and model name of course):

```text
    [[providers]]
    name = "lm_studio"
    api_base = "http://localhost:1234/v1"
    api_key_env_var = ""
    backend = "generic"
```

```text
    [[models]]
    name = "mistralai/ministral-3-3b"
    provider = "lm_studio"
    alias = "lm-studio"
    temperature = 0.7
```

3. Save and restart VSCode
4. Choose your model in the extension and start using it for coding.
   Of course, don't forget to have your model loaded and running in LM Studio and resize the context window as appropriate for your code base.


<img src="images/vscode.png" width="30%" height="30%">


<h2 style="font-size: 25px; font-weight: bold; color: #fb6227;">
  3.4 Local Model from Ollama
</h2>

Ollama is an alternative to LM Studio and can be downloaded on [their website](https://ollama.com/).

- Check that it is correctly installed with the following terminal command
```bash
    ollama --version
```

- Find a suitable model in their [model library](https://ollama.com/search?q=mistral), 'mistral' for example
- Download it from your terminal

```bash
    ollama pull mistral
```

- Check all your available models

```bash
    ollama list
```

- Start the server
```bash
    ollama list
```

- Now you can use it in the Python SDK with the http://localhost:11434 URL

```python
    from mistralai.client import Mistral
    from mistralai.client.models import UserMessage

    mistral = Mistral(
        api_key="dummy",
        server_url="http://localhost:11434"
    )

    response = mistral.chat.complete(
        model="mistral",
        messages=[UserMessage(content="what is the capital of Germany?")]
    )

    print(response.choices[0].message.content)
```

In [1]:
from mistralai.client import Mistral
from mistralai.client.models import UserMessage

mistral = Mistral(
    api_key="dummy",
    server_url="http://localhost:11434"
)

In [2]:
response = mistral.chat.complete(
    model="mistral",
    messages=[UserMessage(content="What is the capital of Germany?")]
)

response.choices[0].message.content

' The capital of Germany is Berlin.'